## Read TAS from nav file for processing CIP in SODA2


In [ ]:
# ---Packages---

import xarray as xr
import glob

from utils.func_nc import sec_since_midnigth
from utils.flight_utils import get_safire_flightid

In [ ]:
# ---Data---
# read in nav files

# -- Paths to datafiles
main_path = '/home/ninalar/Documents/MC2/2022-islas/' # Local disk path to nav data:
pads_path = '/microphy/pads/' # path to pads (CIP and CDP data)
cdp_main_path = main_path + pads_path
path_store = '/home/ninalar/Documents/MC2/Results_2022-islas/TAS/' # where to store the netcdfs # remove when checked

# structure of file names (for access)
file_struct = {'nav_tdyn':'/*_TDYN_*L2_V1.nc',
               'nav_nav': '/*_NAV_*L2_V1.nc'}

flights, safire_to_islas = get_safire_flightid(main_path) # get the flights

In [ ]:
# --- creating TAS file for SODA2 processing ---

for flight in flights:
#for flight in ['as220009']:
    # -- Get NAV files
    # get the nav file from the given flight
    nav_file = glob.glob(main_path + flight + file_struct['nav_tdyn'])

    nav_xds = xr.open_dataset(nav_file[0]) # the nav file xarray
    TAS = nav_xds.TAS1
    flight_id = nav_xds.attrs['flight_id'] # extract flightid for use in filename
    
    # converts the TAS xarray to dataframe with time, LATITUDE, LONGITUDE, ALTITUDE and TAS1 as columns
    # this conversion is done to simplify the creating of the csv file
    tas_df = TAS.to_dataframe()
    tas_df = tas_df.drop(['LATITUDE','LONGITUDE','ALTITUDE'], axis =1) #Need only TAS1

    # add time as a separate column to simplify time manipulation
    tas_df['time'] = tas_df.index

    # recalculate time from datetime to seconds since midnight (func)
    tas_df['time'] = tas_df['time'].apply(sec_since_midnigth)

    # rearrange columns to get correct structure for use in SODA2
    tas_df = tas_df.reindex(columns=['time','TAS1'])
    # set TAS as integer
    #tas_df['TAS1']=tas_df['TAS1'].astype(int)

    # create filename, format: TAS_fightid.csv  (TAS_as220005.csv)
    filename = path_store +'TAS_' + flight_id + '.csv'

    # print tdf to csv-file to be used in SODA2, 
    # change headers to what SODA2 reads,
    # do not add index as correct timeformat is already in column time
    # make sure that it is encoded as ascii
    tas_df.to_csv(filename, header=['time','tas'], encoding='ascii', index=False)


In [10]:
tas_df


,time,TAS1
time,,
2022-03-26 07:57:40.000029,28660.000029,22.900307
2022-03-26 07:57:41.000026,28661.000026,22.830824
2022-03-26 07:57:42.000026,28662.000026,22.882778
2022-03-26 07:57:43.000024,28663.000024,22.859346
2022-03-26 07:57:44.000023,28664.000023,22.842342
...,...,...
2022-03-26 12:33:05.000022,45185.000022,22.461498
2022-03-26 12:33:06.000022,45186.000022,22.576450
2022-03-26 12:33:07.000023,45187.000023,22.639906
